In [1]:
# Put the project directory on the path to make sure we can import PyNitride
import sys, os
sys.path.append(os.path.abspath("../"))

In [2]:
%matplotlib notebook
import numpy as np
from scipy.linalg import norm
from pynitride import ParamDB, Material
from pynitride.bandstuct.kp import kp_bandstructure
from pynitride.crystruct.reciprocal import generate_path
import matplotlib.pyplot as plt
pmdb=ParamDB(units='neu')
to_units=pmdb.to_units
q,hbar,m_e,nm=pmdb.get_constants("e,hbar,m_e,nm")
GaN=Material("GaN",pmdb=pmdb)
pa=np.pi/GaN['lattice.a']

ImportError: No module named 'pynitride.bandstuct'

In [ ]:
GaN=Material("GaN")
AlN=Material("AlN")
a0=GaN['lattice.a']
c0=GaN['lattice.c']
a=AlN['lattice.a']
C13oC33=GaN['stiffness.C13']/GaN['stiffness.C33']
#C13oC33=108/374
st=(a-a0)/a0
sz=-2*C13oC33*st

In [ ]:
kmax=1.5/nm
kvecs=generate_path(points=[[0,0,kmax],[0,0,0],[kmax,0,0]],n=500)
pseudo_k=np.array([norm(kvec)*(-1 if kvec[2]!=0 else 1) for kvec in kvecs])

In [ ]:
bs_r_n=kp_bandstructure(GaN,kvecs,[0,0,0],spin_orbit=False)
bs_r_so=kp_bandstructure(GaN,kvecs,[0,0,0],spin_orbit=True)
bs_s_n=kp_bandstructure(GaN,kvecs,[st,st,sz],spin_orbit=False)
bs_s_so=kp_bandstructure(GaN,kvecs,[st,st,sz],spin_orbit=True)

In [ ]:
def plot_bs(pseudo_k,bs,label='',color='',new_fig=True):
    if new_fig:
        plt.figure(figsize=(6,6))
    plt.subplot(212)

    plt.plot(pseudo_k/pa,bs[:,:6],color=color,label=label)
    plt.ylim(-.05,.0)
    plt.ylim(-.9,.1)
    plt.xlim(-.15,.15)
    plt.xlabel('            $\\leftarrow k_z, k_x\\rightarrow$   [$\pi/a_0$]')
    plt.xticks([-.15,.15])
    plt.ylabel('$E-E_{V0}$ [eV]')
    
    plt.subplot(211)
    plt.plot(pseudo_k/pa,bs[:,6:],color=color,label=label)
    plt.xticks([])
    plt.ylabel('$E-E_{C0}$ [eV]')
    plt.xlim(-.15,.15)
    #plt.ylim(0,.05)
    plt.tight_layout()

In [ ]:
plot_bs(pseudo_k,bs_r_n,color='b')
plot_bs(pseudo_k,bs_r_so,color='r',new_fig=False)
plt.title("Relaxed: SO versus no SO")

In [ ]:
plot_bs(pseudo_k,bs_s_n,color='b')
plot_bs(pseudo_k,bs_s_so,color='r',new_fig=False)
plt.title("Strained: SO versus no SO")

In [ ]:
plot_bs(pseudo_k,bs_r_so,color='b')
plot_bs(pseudo_k,bs_s_so,color='r',new_fig=False)
plt.title("Relaxed vs Strained, SO")

In [ ]:
plot_bs(pseudo_k,bs_r_n,color='b')
plot_bs(pseudo_k,bs_s_n,color='r',new_fig=False)
plt.title("Relaxed vs Strained, no SO")

In [ ]:
def print_eff_mass(bs,k,p):
    n=len(k)
    for i in reversed(2*np.arange(3)):
        x=("E0={:7.3f}".format(bs[0,i]))
        x+=("     m={:7.3f}".format(-1/(2*m_e/hbar**2*(bs[p,i]-bs[0,i])/norm(k[p])**2)))
        print(x)

unstrained
---

In [ ]:
print("no SO")
print("x")
print_eff_mass(np.flipud(bs_r_n[:500]),np.flipud(kvecs[:500]),5)
print("y")
print_eff_mass((bs_r_n[499:]),(kvecs[499:]),5)
print("\n")
print("SO")
print("x")
print_eff_mass(np.flipud(bs_r_so[:500]),np.flipud(kvecs[:500]),5)
print("y")
print_eff_mass((bs_r_so[499:]),(kvecs[499:]),5)

strained
---

In [ ]:
print("no SO")
print("x")
print_eff_mass(np.flipud(bs_s_n[:500]),np.flipud(kvecs[:500]),5)
print("y")
print_eff_mass((bs_s_n[499:]),(kvecs[499:]),5)
print("\n")
print("SO")
print("x")
print_eff_mass(np.flipud(bs_s_so[:500]),np.flipud(kvecs[:500]),5)
print("y")
print_eff_mass((bs_s_so[499:]),(kvecs[499:]),5)

strained at 100
---

In [ ]:
print("SO")
print("x")
print_eff_mass(np.flipud(bs_s_so[:500]),np.flipud(kvecs[:500]),100)
print("y")
print_eff_mass((bs_s_so[499:]),(kvecs[499:]),100)

In [ ]:
bs_s_so[499][6]-bs_s_so[499][5]

In [ ]:
bs_r_so[499][6]-bs_r_so[499][5]

In [ ]:
bs_s_so[499][6]-bs_r_so[499][6]

In [ ]:
Material("AlN")["Eg"]

In [ ]:
Material("AlN")["electron.DEc"]-(bs_s_so[499][6]-bs_r_so[499][6])

AlN
===

In [ ]:
aln_bs_r_so=kp_bandstructure(AlN,kvecs,[0,0,0],spin_orbit=True)

In [ ]:
plot_bs(pseudo_k,aln_bs_r_so,color='r')
#plot_bs(pseudo_k,bs_r_so,color='r',new_fig=False)
plt.title("Relaxed: SO")# versus no SO")

In [ ]:
print("SO")
print("x")
print_eff_mass(np.flipud(aln_bs_r_so[:500]),np.flipud(kvecs[:500]),100)
print("z")
print_eff_mass((aln_bs_r_so[499:]),(kvecs[499:]),100)